In the notebook ```sentence_encoder_test.ipynb``` we did some basic visualization of encoding the distributions of winners vs nonwinners. However, the plots that we got seem to indicate that there was not much substantial difference between the winners and nonwinners. 

In order to test if there is any actual difference, we will perform some hypothesis test.

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sentence_transformers import SentenceTransformer

transformer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
books = pd.read_csv('../../data/final_book_dataset_cleaned.csv', sep = '\t', 
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
})

books = books.dropna() # we do this in order to remove potential outliers that may skew our data incorrectly

In [5]:
sample = books[(2000 <= books.release_year) & (books.release_year <= 2013)]
sample

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
30045,741561,Bad Dreams,Anne Fine,2000,2000-06-00,Delacorte Press,1947,"Leicester, Leicestershire, England, UK",0385327579,"Despite her preference for books over friends,...",[],False,False
30049,918449,Mr. Mee,Andrew Crumey,2000,2000-05-00,Picador,1961,"Kirkintilloch, Dunbartonshire, Scotland, UK",0330376802,A New York Times Notable Book of the Year In t...,[],False,False
30053,739969,Whispers in the Sand,Barbara Erskine,2000,2000-09-00,HarperCollins (UK),1944,"Nottingham, Nottinghamshire, England, UK",000225784X,"Recently divorced, Anna decides to cheer herse...","['Fiction', 'Fantasy', 'Adventure', 'Romance',...",False,False
30056,729397,In the Country of the Young,Lisa Carey,2000,2000-11-00,William Morrow / HarperCollins,1970,"Boston, Massachusetts, USA",0380976757,"On a stormy November night in 1848, a ship car...","['Fantasy', 'Fiction', 'History']",False,False
30072,873545,Trouble on Tattooine,Dave Wolverton,2000,2000-04-00,Scholastic,1957,"Springfield, Oregon, USA",043910145X,Anakin Skywalker and his friends are in big tr...,"[""Children's stories"", 'Adventure']",False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73334,1633699,Strikeforce,Nick James,2013,2013-10-08,Flux,1983,"Portland, Oregon, USA",9780738736372.0,As the alien Authority attacks Earth using a t...,['young-adult sf'],False,False
73341,1601405,Storm Force,Susannah Sandlin,2013,2013-03-19,Montlake Romance,1956,"Winfield, Alabama, USA",9781477857571.0,When a bomb explodes inside a Houston high-ris...,"['paranormal romance', 'paranormal suspense', ...",False,False
73342,1632034,Treecat Wars,"Jane Lindskold, David Weber",2013,2013-10-01,Baen Books,1962,"Washington, District of Columbia, USA",9781451639339.0,"The fires are out, but the trouble’s just begi...","['science fiction', 'young-adult sf', 'Science...",False,False
73345,1600519,Dragonwitch,Anne Elisabeth Stengl,2013,2013-07-15,Bethany House Publishers,1986,"McChord AFB, Washington, USA",9780764210273.0,A New Tale Is Added to this Christy Award-Winn...,"['religious fantasy', 'young-adult fantasy']",False,False


In [6]:
locus = sample[sample.locus]
nonlocus = sample[~(sample.locus)]

hugo = sample[sample.hugo]
nonhugo = sample[~(sample.hugo)]

In [7]:
locus_embed = transformer.encode(np.array(locus.book_synopsis))

In [8]:
locus_embed

array([[-0.1016467 ,  0.00948468, -0.01626917, ..., -0.14832212,
         0.03050089,  0.06897663],
       [-0.06895619,  0.1414698 ,  0.01198734, ...,  0.00482523,
        -0.03022195,  0.04948013],
       [ 0.07926933, -0.0495548 ,  0.02950573, ...,  0.02366009,
        -0.00610185,  0.06315764],
       ...,
       [-0.02824043,  0.02276585,  0.01558276, ..., -0.02006578,
         0.05978588,  0.05705544],
       [-0.09500595,  0.04716651, -0.04036311, ..., -0.03036789,
        -0.05574731, -0.03680762],
       [-0.07830553,  0.00356592, -0.05418875, ..., -0.05396573,
         0.03705452,  0.04993296]], shape=(407, 384), dtype=float32)

In [9]:
nonlocus_embed = transformer.encode(np.array(nonlocus.book_synopsis))

In [10]:
nonlocus_embed

array([[-0.02888518, -0.05045382,  0.09129327, ...,  0.07984006,
        -0.04837747, -0.01610984],
       [-0.09443662, -0.04422923,  0.00704636, ..., -0.02314086,
         0.00194414, -0.01518036],
       [-0.10260569,  0.07463712,  0.05064687, ...,  0.04494895,
        -0.01507652, -0.04994804],
       ...,
       [ 0.03713531,  0.02617711,  0.05042417, ..., -0.07394157,
        -0.00733219,  0.00755598],
       [-0.03616722,  0.07856765,  0.0079909 , ..., -0.12063679,
         0.02614661,  0.00224246],
       [-0.00961152,  0.04051654, -0.03969164, ..., -0.05296772,
        -0.06586705,  0.07554311]], shape=(8217, 384), dtype=float32)

As a first sanity check, we should check to see if the difference between the sample means of these two arrays are sufficiently apart.

In [11]:
locus_mean = locus_embed.mean(axis=0)
nonlocus_mean = nonlocus_embed.mean(axis=0)

In [12]:
np.sqrt(
    np.square(locus_mean - nonlocus_mean).sum()
    )

np.float32(0.12571232)

This difference seems reasonably apart, so let's try to proceed to a hypothesis test.

We have two samples ```locus_embed``` and ```nonlocus_embed```. We want to see if these two samples come from the same multivariate distribution. It seems that one way to do this is to apply Hotelling's $T^2$ hypothesis test.

We assume somehow that the result of our embeddings are normal distributions, this test statistics should be valid enough to determine if these two distributions are actually different. We first apply this test naively, and then maybe later we can try some bootstrapping method.

In [ ]:
#%pip install scikit-fda # run once to install scikit-fda which implements the T^2 test

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 41.2 MB/s  0:00:0058.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [scikit-fda]0m  9/10 [scikit-fda]sets]
Note: you may need to restart the kernel to use updated packages.


In [16]:
from skfda import FDataGrid
from skfda.inference.hotelling import hotelling_test_ind

In [17]:
fda_locus = FDataGrid(locus_embed)
fda_nonlocus = FDataGrid(nonlocus_embed)

In [ ]:
# this takes a very long time to run 
# on my desktop machine (7900X3d with 32GB RAM DDR5) it took around 9min
t2_stat, p_val = hotelling_test_ind(fda_locus, fda_nonlocus, n_reps = 10_000)

print("T^2", t2_stat)
print('p value', p_val)

T^2 1051.809413246854
p value 0.0


Do the same for the Hugo vs non-Hugo awards

In [19]:
hugo_embed = transformer.encode(np.array(hugo.book_synopsis))
nonhugo_embed = transformer.encode(np.array(nonhugo.book_synopsis))

In [20]:
fda_hugo = FDataGrid(hugo_embed)
fda_nonhugo = FDataGrid(nonhugo_embed)

In [21]:
# again will take a while to run
t2_stat, p_val = hotelling_test_ind(fda_hugo, fda_nonhugo, n_reps = 10_000)

print("T^2", t2_stat)
print('p val', p_val)

T^2 535.0193956282433
p val 0.0003


After running these hypothesis tests with ```n_reps=1_000_000```, we found the following results:

Hugo Hotelling $T^2$ hypothesis testing

Permutation testing with 1,000,000 repetitions

$T^2$ statistic: 535.0193956282445

$p$ value: 0.000167

Locus Hotelling $T^2$ hypothesis testing

Permutation testing with 1,000,000 repetitions

$T^2$ statistic: 1051.8094132468555

$p$ value: 0.0